In [14]:
import openai, json, requests

client = openai.OpenAI()
messages = []

In [15]:
def get_popular_movies():
    url = "https://nomad-movies.nomadcoders.workers.dev/movies"
    response = requests.get(url)
    return response.json()

def get_movie_details(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}"
    response = requests.get(url)
    return response.json()

def get_similar_movies(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/similar"
    response = requests.get(url)
    return response.json()

FUNCTION_MAP = {
    "get_popular_movies" : get_popular_movies,
    "get_movie_details" : get_movie_details,
    "get_similar_movies" : get_similar_movies
}
    

In [16]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get the list of popular movies",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get the details of a movie",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer"}},
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Get the list of similar movies",
            "parameters": {
                "type": "object",
                "properties": {"id": {"type": "integer"}},
                "required": ["id"]
            }
        }
    }
]


In [17]:
from openai.types.chat import ChatCompletionMessage


def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }
                for tool_call in message.tool_calls
            ],
        })
        
        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            function_args = tool_call.function.arguments

            print(f"Running {function_name} with {function_args}", flush=True)

            try:
                args = json.loads(function_args)
            except json.JSONDecodeError:
                args = {}
            
            function_to_run = FUNCTION_MAP[function_name]
            result = function_to_run(**args)
            
            print(f"Ran {function_name} with {args} and got {result}", flush=True)
            messages.append({
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id
            })
        call_ai()
    else:
        content = message.content or ""
        messages.append({"role": "assistant", "content": content})
        print(f"AI: {content}", flush=True)

def call_ai():
    response = client.chat.completions.create(
            model = "gpt-4o-mini",
            messages = messages,
            tools = TOOLS,
    )
    process_ai_response(response.choices[0].message)
    


In [18]:
while True:
    print("", flush=True)  # 이전 출력과 구분, 버퍼 비우기
    message = input("Send a message to the LLM (q/quit to exit): ")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}", flush=True)
        call_ai()
        
        

User: 지금 인기 있는 영화 알려줘
Running get_popular_movies with {}
Ran get_popular_movies with {} and got [{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/6YjnTRBz704LF1uJ3ZC4wsS9T8r.jpg', 'genre_ids': [28, 80, 53], 'id': 1290821, 'original_language': 'en', 'original_title': 'Shelter', 'overview': 'A man living in self-imposed exile on a remote island rescues a young girl from a violent storm, setting off a chain of events that forces him out of seclusion to protect her from enemies tied to his past.', 'popularity': 434.2537, 'poster_path': 'https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg', 'release_date': '2026-01-28', 'title': 'Shelter', 'video': False, 'vote_average': 6.94, 'vote_count': 117}, {'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/7HKpc11uQfxnw0Y8tRUYn1fsKqE.jpg', 'genre_ids': [878, 28, 53], 'id': 1236153, 'original_language': 'en', 'original_title': 'Mercy', 'overview': 'In the near future, a detective stands on trial accused 